# Classificação Quântica com o Dataset Pima Indians Diabetes
### Da preparação dos dados até a execução em um computador quântico da IBM

Este notebook implementa um **VQC (Variational Quantum Classifier)** aplicado ao dataset **Pima Indians Diabetes** para prever se uma paciente tem diabetes (`1`) ou não (`0`).

### O que é um VQC?

Um VQC é o equivalente quântico de uma rede neural simples, com três partes:
1. **Feature map** (embedding): transforma dados clássicos em estados quânticos
2. **Ansatz**: circuito com parâmetros treináveis (os "pesos" do modelo)
3. **Medicao**: lê o resultado do circuito e converte em previsão

O treinamento acontece classicamente: um otimizador ajusta os parâmetros do circuito para minimizar o erro nas previsões.

### Por que Diabetes e mais interessante que Iris?

| | Iris (exemplo base) | Diabetes (este notebook) |
|---|---|---|
| Separabilidade | Alta | Moderada — classes se sobrepoem |
| Desbalanceamento | Nenhum | 65% / 35% |
| Dados ausentes | Nenhum | Zeros fisiologicos invalidos |
| Relevância | Pedagogica | Aplicacao clinica real |

> **Versoes:** Qiskit 2.x · qiskit-ibm-runtime 0.4x · scikit-learn 1.x


## 0. Preparacao do ambiente

In [ ]:
import sys
# Instala no mesmo Python que o notebook usa (evita conflitos de ambiente)
!{sys.executable} -m pip install qiskit qiskit-ibm-runtime scikit-learn seaborn matplotlib pylatexenc -q


In [ ]:
import numpy as np          # operacoes numericas e algebra linear
import pandas as pd         # manipulacao de tabelas (DataFrames)
import matplotlib.pyplot as plt  # graficos
import seaborn as sns       # visualizacoes estatisticas

import qiskit               # framework de computacao quantica
import qiskit_ibm_runtime   # conexao com hardware IBM

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_SEED = 42            # semente para reproducibilidade
np.random.seed(RANDOM_SEED)

print("qiskit      :", qiskit.__version__)
print("ibm-runtime :", qiskit_ibm_runtime.__version__)


## 1. Entendendo os dados

O **Pima Indians Diabetes Dataset** tem 768 pacientes do sexo feminino com pelo menos 21 anos.

| Atributo | Descricao |
|---|---|
| Pregnancies | Numero de gestacoes |
| Glucose | Concentracao de glicose plasmatica (mg/dL) |
| BloodPressure | Pressao arterial diastolica (mm Hg) |
| SkinThickness | Espessura da dobra cutanea do triceps (mm) |
| Insulin | Insulina serica de 2h (mu U/ml) |
| BMI | Indice de massa corporal (kg/m2) |
| DiabetesPedigreeFunction | Historico familiar de diabetes |
| Age | Idade (anos) |
| **Outcome** | **1 = diabetica / 0 = nao-diabetica** |


In [ ]:
# Carrega o CSV — coloque o arquivo diabetes.csv na mesma pasta do notebook
df = pd.read_csv("diabetes.csv")
print("Shape do dataset:", df.shape)  # esperado: (768, 9)
df.head(10)


In [ ]:
print("--- Resumo estatistico ---")
display(df.describe().round(2))

print("\nDistribuicao das classes:")
print(df["Outcome"].value_counts())
print(f"\nDesbalanceamento: {df['Outcome'].mean()*100:.1f}% positivos (diabetico)")
# Nota: 65% nao-diabetico / 35% diabetico
# Com classes desbalanceadas, acuracia so nao e suficiente para avaliar o modelo


### 1.1 Zeros fisiologicamente impossiveis

Cinco atributos nao podem ser zero: `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` e `BMI`.
Zeros nessas colunas sao **valores ausentes codificados** — uma limitacao do dataset original.


In [ ]:
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("Zeros por coluna (valores ausentes disfarçados):")
for c in zero_cols:
    pct = (df[c] == 0).mean() * 100
    print(f"  {c:30s}: {(df[c]==0).sum():3d} ({pct:.1f}%)")


In [ ]:
# Estrategia: substituir 0 por NaN, depois imputar com a MEDIANA da mesma classe
# Por que mediana? E robusta a outliers (insulina tem distribuicao muito assimetrica)
# Por que por classe? Diabeticas e nao-diabeticas tem perfis fisiologicos diferentes

df_clean = df.copy()
for col in zero_cols:
    df_clean[col] = df_clean[col].replace(0, float("nan"))
    df_clean[col] = df_clean.groupby("Outcome")[col].transform(
        lambda s: s.fillna(s.median())
    )

print("Valores ausentes apos imputacao:", df_clean.isnull().sum().sum())
print("Shape final:", df_clean.shape)


### 1.2 Analise exploratoria visual

Os graficos mostram a distribuicao de cada atributo separada por classe.
Atributos com pouca sobreposicao entre as curvas sao melhores candidatos para o circuito quantico.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
features = [c for c in df_clean.columns if c != "Outcome"]
for ax, feat in zip(axes.flat, features):
    sns.kdeplot(data=df_clean, x=feat, hue="Outcome", ax=ax,
                fill=True, alpha=0.4, palette={0: "steelblue", 1: "tomato"})
    ax.set_title(feat)
fig.suptitle("Distribuicao por classe | azul = nao-diabetico | vermelho = diabetico", y=1.01)
plt.tight_layout()
plt.show()
# Observe: Glucose e BMI mostram as maiores separacoes entre as classes


In [ ]:
# Correlacao de Pearson: mede a relacao linear entre cada atributo e o Outcome
# Quanto mais alto, mais o atributo "explica" o diagnostico
corr = df_clean.corr()["Outcome"].drop("Outcome").sort_values(ascending=False)

plt.figure(figsize=(7, 4))
sns.barplot(x=corr.values, y=corr.index, palette="coolwarm_r")
plt.axvline(0, color="k", linewidth=0.8)
plt.title("Correlacao de Pearson com Outcome")
plt.xlabel("Correlacao")
plt.tight_layout()
plt.show()

print("Top 2 atributos:", list(corr.index[:2]))
# Resultado esperado: Glucose e BMI — escolhemos esses para o circuito


## 2. Preparacao dos dados para o VQC

Cada decisao tem uma justificativa tecnica:

**1. Dois atributos (`Glucose` e `BMI`)**
Com 2 features usamos apenas **2 qubits**. Circuito raso = menos erro em hardware real.
Tambem conseguimos desenhar a fronteira de decisao em 2D.

**2. Rotulos em {-1, +1}**
O observavel Z-tensor-Z tem autovalores exatamente +/-1, que casam com nossos rotulos.

**3. Normalizar para [0, pi]**
Os atributos viram **angulos de rotacao** no circuito (porta RY).
Manter em [0, pi] evita ambiguidade: rotacoes de 0 e 2*pi sao identicas.

**4. Split estratificado**
Preserva a proporcao 65%/35% tanto no treino quanto no teste.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# (1) Selecionar Glucose e BMI
FEATURES = ["Glucose", "BMI"]
X_all = df_clean[FEATURES].values
y_all = df_clean["Outcome"].values

# (2) Converter rotulos: 0 -> -1  (nao-diabetica = -1, diabetica = +1)
y_all = np.where(y_all == 1, 1, -1)

# (3) Normalizar cada feature para o intervalo [0, pi]
scaler = MinMaxScaler(feature_range=(0, float(np.pi)))
X_all = scaler.fit_transform(X_all)

# (4) Split estratificado (preserva proporcao das classes)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_all
)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print(f"Intervalo de X apos normalizacao: {X_all.min():.3f} -> {X_all.max():.3f}")
print(f"Classes no treino: {dict(zip(*np.unique(y_train, return_counts=True)))}")


## 3. Modos de embedding (codificacao dos dados)

Para um circuito quantico processar dados classicos, precisamos **codifica-los como estados quanticos**.

| Embedding | Entrelaçamento | Profundidade | Quando usar |
|---|---|---|---|
| Angulo (RY) | Nao | Rasa | Dados simples, hardware limitado |
| z_feature_map | Nao | Media | Boa base sem correlacoes entre qubits |
| zz_feature_map | Sim (ZZ) | Profunda | Dados com estrutura nao-linear complexa |


In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import z_feature_map, zz_feature_map

# Codificacao por angulo (a mais simples):
# Cada atributo x[i] vira o angulo de uma rotacao RY no qubit i
# RY(angulo)|0> = cos(angulo/2)|0> + sin(angulo/2)|1>
# Sem entrelaçamento — os qubits sao independentes
x = ParameterVector("x", 2)
angle_map = QuantumCircuit(2, name="AngleEncoding")
angle_map.ry(x[0], 0)   # qubit 0 recebe o angulo de Glucose
angle_map.ry(x[1], 1)   # qubit 1 recebe o angulo de BMI

print("=== Codificacao por Angulo ===")
angle_map.draw("mpl")


In [ ]:
# z_feature_map: acrescenta Hadamards + rotacoes de fase
# Hadamard: coloca o qubit em superposicao |0> -> (|0>+|1>)/sqrt(2)
# Ainda sem entrelaçamento entre qubits
print("=== z_feature_map ===")
z_map = z_feature_map(feature_dimension=2, reps=1, parameter_prefix="x")
z_map.decompose().draw("mpl")


In [ ]:
# zz_feature_map: adiciona portas de entrelaçamento ZZ entre os qubits
# Cria correlacoes quanticas entre os qubits — "mais quantico"
# Mas: circuito mais profundo = mais sensivel a ruido no hardware real
print("=== zz_feature_map ===")
zz_map = zz_feature_map(feature_dimension=2, reps=1, entanglement="linear", parameter_prefix="x")
zz_map.decompose().draw("mpl")


### 3.1 Comparando embeddings pelo kernel quantico

O kernel quantico mede a similaridade entre duas amostras como a fidelidade entre seus estados:

    K(xi, xj) = |<phi(xi)|phi(xj)>|^2

Um embedding util para classificacao deve mostrar **blocos** na matriz:
- Alta similaridade (~1) entre amostras da mesma classe
- Baixa similaridade (~0) entre amostras de classes diferentes


In [ ]:
from qiskit.quantum_info import Statevector

def class_balanced_subset(X, y, per_class=6):
    # Seleciona `per_class` amostras de cada classe para visualizacao
    idx = np.concatenate([np.where(y == -1)[0][:per_class],
                          np.where(y == +1)[0][:per_class]])
    return X[idx], y[idx]

def fidelity_kernel(feature_map, data):
    # Calcula K[i,j] = |<phi(xi)|phi(xj)>|^2 para todos os pares
    # K[i,i] = 1.0 sempre (um estado e identico a si mesmo)
    # K[i,j] proximo de 1 -> amostras similares no espaco quantico
    # K[i,j] proximo de 0 -> amostras diferentes no espaco quantico
    states = [Statevector(feature_map.assign_parameters(row)) for row in data]
    n = len(states)
    K = np.empty((n, n))
    for i in range(n):
        for j in range(n):
            K[i, j] = float(np.abs(states[i].inner(states[j])) ** 2)
    return K

Xk, yk = class_balanced_subset(X_train, y_train, per_class=6)

K_z  = fidelity_kernel(z_feature_map(2, reps=1, parameter_prefix="x"), Xk)
K_zz = fidelity_kernel(zz_feature_map(2, reps=1, entanglement="linear", parameter_prefix="x"), Xk)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
labels = [("(N%d)" % i if v < 0 else "(D%d)" % i) for i, v in enumerate(yk)]

for ax, K, title in [(axes[0], K_z, "z_feature_map"), (axes[1], K_zz, "zz_feature_map")]:
    sns.heatmap(K, ax=ax, cmap="viridis", vmin=0, vmax=1, square=True,
                xticklabels=labels, yticklabels=labels,
                cbar_kws={"label": "fidelidade"})
    ax.set_title("Kernel quantico — " + title)
    ax.axhline(6, color="red", linewidth=1.5, linestyle="--")
    ax.axvline(6, color="red", linewidth=1.5, linestyle="--")

plt.suptitle("N = nao-diabetica  |  D = diabetica  |  linha vermelha = divisao entre classes", fontsize=10)
plt.tight_layout()
plt.show()


## 4. O ansatz (circuito variacional)

O ansatz e a parte **treinavel** do circuito. Contem parametros theta que o otimizador vai ajustar — como os pesos de uma rede neural.

**Estrutura geral do classificador:**

    f_theta(x) = <0| U_dag(x) W_dag(theta) O W(theta) U(x) |0>

- U(x): feature map — codifica os dados
- W(theta): ansatz — bloco treinavel
- O: observavel (ZZ) — o que medimos
- Resultado: numero em [-1, +1] -> nossa previsao

**Por que efficient_su2 com reps=2?**
Camadas de RY e RZ intercaladas com CNOT.
`reps=2` = 12 parametros treinaveis: expressivo sem cair em *barren plateaus*.
*Barren plateaus*: gradientes que somem com circuitos muito profundos, impossibilitando o treinamento.


In [ ]:
from qiskit.circuit.library import efficient_su2

# parameter_prefix="theta" e CRITICO:
# O Qiskit ordena parametros ALFABETICAMENTE.
# Como 'x' < 'theta' na ordem Unicode, os dados vem ANTES dos pesos.
# Isso garante que o forward() concatene os valores na ordem certa.
ansatz = efficient_su2(
    num_qubits=2,
    reps=2,                    # 2 repeticoes -> 12 parametros treinaveis
    entanglement="linear",     # CNOT entre qubits vizinhos
    parameter_prefix="theta"   # nome dos parametros treinaveis
)

print("Parametros treinaveis do ansatz:", ansatz.num_parameters)
ansatz.decompose().draw("mpl")


## 5. Montando o VQC — Passo 1 dos Qiskit Patterns

**Qiskit Patterns** e o fluxo IBM para hardware quantico:
1. **Mapear** -> criar o circuito (esta secao)
2. **Otimizar** -> transpilar para o hardware
3. **Executar** -> rodar com primitives
4. **Pos-processar** -> analisar resultados

Combinamos feature map + ansatz em um unico circuito.

> **Detalhe critico:** o Qiskit ordena parametros **alfabeticamente**.
> Por isso usamos `x` para dados e `theta` para pesos.
> Como `x < theta`, os dados sempre vem primeiro.
> O `assert` abaixo verifica isso automaticamente.


In [ ]:
from qiskit.quantum_info import SparsePauliOp

num_qubits = 2

# Feature map: codifica Glucose e BMI como rotacoes no circuito
feature_map = z_feature_map(feature_dimension=num_qubits, reps=1, parameter_prefix="x")

# Monta o circuito: feature_map DEPOIS ansatz em sequencia
circuit = QuantumCircuit(num_qubits)
circuit.compose(feature_map, inplace=True)   # primeiro: codifica os dados
circuit.compose(ansatz, inplace=True)         # depois: aplica os pesos treinaveis

# Observavel ZZ: autovalores +1 e -1, que mapeiam para nossas classes
# "ZZ" = medir Z no qubit 0 E Z no qubit 1 simultaneamente
observable = SparsePauliOp.from_list([("ZZ", 1)])

n_input  = feature_map.num_parameters   # 2 (Glucose e BMI)
n_weight = ansatz.num_parameters        # 12 (pesos treinaveis)
print(f"Parametros de dados:    {n_input}")
print(f"Parametros treinaveis:  {n_weight}")
print(f"Total no circuito:      {n_input + n_weight}")

# Verificacao de seguranca
sorted_names = [p.name for p in circuit.parameters]
assert all(n.startswith("x")     for n in sorted_names[:n_input]),  sorted_names
assert all(n.startswith("theta") for n in sorted_names[n_input:]),  sorted_names
print("Ordem verificada: dados (x) antes dos pesos (theta) OK")

# Visualiza o circuito completo
circuit.decompose().draw("mpl", fold=-1)


## 6. Treinamento no simulador — Passo 3 dos Qiskit Patterns

### Como funciona o treinamento?

1. `forward()` recebe uma amostra [Glucose, BMI normalizados] e os pesos theta
2. Monta o vetor [dados, pesos] e passa ao `StatevectorEstimator`
3. O Estimator executa o circuito e retorna o valor esperado <ZZ> em [-1, +1]
4. `mse_cost()` calcula o MSE entre previsoes e rotulos reais
5. O otimizador COBYLA ajusta os pesos para minimizar o MSE

### Por que COBYLA?
Otimizador sem gradiente — nao precisa calcular derivadas do circuito.
Para circuitos rasos com poucos parametros, converge bem e e robusto.

### Nota sobre tempo de execucao
Esta configuracao usa 50 amostras e 50 iteracoes para rodar em ~30-60 segundos.
Para producao: aumente para 576 amostras e 300 iteracoes (20-60 minutos).


In [ ]:
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize

# StatevectorEstimator: simulador EXATO (sem ruido)
# Calcula o valor esperado diretamente do vetor de estado
estimator = StatevectorEstimator()

# Subsample para treino rapido
# 50 amostras em vez de 576 -> de 20-60 min para ~30-60 segundos
# O fluxo e identico — so o tempo e a acuracia mudam
idx = np.random.choice(len(X_train), size=50, replace=False)
X_train_small = X_train[idx]
y_train_small = y_train[idx]

def forward(data, weights):
    # Calcula <ZZ> para cada amostra em `data`
    # Parametros:
    #   data:    matriz (n_amostras, 2) com Glucose e BMI normalizados
    #   weights: vetor (12,) com os pesos treinaveis
    # Retorna:
    #   array (n_amostras,) com valores em [-1, +1]
    #   +1 -> preve diabetica | -1 -> preve nao-diabetica
    data = np.atleast_2d(data)
    w = np.broadcast_to(weights, (data.shape[0], len(weights)))
    # Cada linha fica: [Glucose, BMI, theta0, theta1, ..., theta11]
    params = np.concatenate([data, w], axis=1)
    pub = (circuit, observable, params)   # PUB = Primitive Unified Bloc
    result = estimator.run([pub]).result()[0]
    return np.asarray(result.data.evs)    # evs = expected values

def mse_cost(weights):
    # MSE = (1/n) * soma((y_previsto - y_real)^2)
    # Quanto menor o MSE, mais o circuito acerta
    preds = forward(X_train_small, weights)
    cost = float(np.mean((preds - y_train_small) ** 2))
    history.append(cost)  # guarda para plotar a curva de aprendizado
    return cost

def predict(data, weights):
    # Converte saida continua em classe binaria:
    # <ZZ> >= 0 -> +1 (diabetica) | <ZZ> < 0 -> -1 (nao-diabetica)
    return np.where(forward(data, weights) >= 0, 1, -1)

# Treinamento
history = []
np.random.seed(RANDOM_SEED)
# Inicializa pesos aleatoriamente em [0, 2*pi] (angulos de rotacao)
initial_weights = np.random.rand(n_weight) * 2 * float(np.pi)

print("Iniciando treinamento... (30-60 segundos)")
result = minimize(
    mse_cost,
    initial_weights,
    method="COBYLA",
    options={"maxiter": 50, "rhobeg": 0.5},
)
trained_weights = result.x
print(f"Custo final: {result.fun:.4f} | Iteracoes: {len(history)}")


In [ ]:
# Curva de aprendizado: mostra como o custo cai ao longo das iteracoes
# Uma curva que cai e estabiliza = convergencia saudavel
plt.figure(figsize=(9, 4))
sns.lineplot(x=range(len(history)), y=history, color="steelblue")
plt.xlabel("Iteracao")
plt.ylabel("Custo (MSE)")
plt.title("Convergencia do treinamento — VQC no dataset Diabetes")
plt.tight_layout()
plt.show()


## 7. Avaliacao — Passo 4 dos Qiskit Patterns

### Metricas usadas

**Acuracia:** percentual de previsoes corretas. Facil de entender, mas enganosa com dados desbalanceados.

**Sensibilidade (Recall para diabeticos):** dos pacientes realmente diabeticos, quantos o modelo identificou?
Em contexto clinico, falsos negativos sao perigosos — um diabetico nao diagnosticado nao recebe tratamento.

**Matriz de confusao:**

| | Previsto: nao-diabetica | Previsto: diabetica |
|---|---|---|
| Real: nao-diabetica | Verdadeiro Negativo (OK) | Falso Positivo |
| Real: diabetica | Falso Negativo (PERIGO) | Verdadeiro Positivo (OK) |


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

preds_train = predict(X_train, trained_weights)
preds_test  = predict(X_test,  trained_weights)

print(f"Acuracia treino : {accuracy_score(y_train, preds_train)*100:.1f}%")
print(f"Acuracia teste  : {accuracy_score(y_test,  preds_test )*100:.1f}%")
print()
print("--- Relatorio completo (conjunto de teste) ---")
print(classification_report(y_test, preds_test,
                            target_names=["nao-diabetica (-1)", "diabetica (+1)"]))
# precision: dos que o modelo disse diabetico, quantos realmente eram?
# recall:    dos realmente diabeticos, quantos o modelo encontrou?
# f1-score:  media harmonica de precision e recall


In [ ]:
cm = confusion_matrix(y_test, preds_test, labels=[-1, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["nao-diabetica", "diabetica"],
            yticklabels=["nao-diabetica", "diabetica"])
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confusao — conjunto de teste")
plt.tight_layout()
plt.show()


In [ ]:
# Fronteira de decisao em 2D
# Avalia o classificador em uma grade de 50x50 pontos no espaco [0,pi]^2
# A linha preta (contorno em 0) e a FRONTEIRA DE DECISAO
# Pontos acima: previsto diabetica | Pontos abaixo: previsto nao-diabetica

gx, gy = np.meshgrid(np.linspace(0, float(np.pi), 50),
                     np.linspace(0, float(np.pi), 50))
grid = np.c_[gx.ravel(), gy.ravel()]
zz = forward(grid, trained_weights).reshape(gx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(gx, gy, zz, levels=20, cmap="coolwarm", alpha=0.65)
plt.colorbar(label="<ZZ>  (vermelho = diabetica | azul = nao-diabetica)")
plt.contour(gx, gy, zz, levels=[0.0], colors="k", linewidths=2)

for label, name, marker, color in [
    (-1, "nao-diabetica", "o", "steelblue"),
    ( 1, "diabetica",     "s", "tomato")
]:
    tr = y_train == label
    te = y_test  == label
    # Pontos solidos = treino | contornos vazios = teste
    plt.scatter(X_train[tr,0], X_train[tr,1], marker=marker, color=color,
                edgecolor="k", s=50, label=name + " (treino)")
    plt.scatter(X_test[te,0],  X_test[te,1],  marker=marker, color="none",
                edgecolor=color, s=80, linewidths=1.8, label=name + " (teste)")

plt.xlabel("Glucose (normalizado para [0, pi])")
plt.ylabel("BMI (normalizado para [0, pi])")
plt.title("Fronteira de Decisao do VQC — Diabetes (Glucose x BMI)")
plt.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## 8. Execucao no computador quantico real da IBM

Ate aqui rodamos no simulador exato (sem ruido, sem fila).
Para hardware real, precisamos de dois passos extras:

**Passo 2 — Transpilar:** converter o circuito para as portas nativas e topologia fisica do chip.
Cada computador IBM tem suas proprias portas e conexoes entre qubits.

**Passo 3 — Executar com Runtime:** usar `EstimatorV2` do `qiskit-ibm-runtime`.

### Estrategia
Usamos os pesos ja treinados no simulador para fazer inferencia no hardware.
Nao retreinamos — economiza tempo de QPU e isola o efeito do ruido.

### Como obter o token IBM
1. Crie conta em quantum.cloud.ibm.com
2. Copie o API Token da sua conta
3. Cole no lugar de SEU_TOKEN_AQUI e execute uma vez


In [ ]:
# Execute esta celula UMA VEZ para salvar as credenciais localmente
# Depois comente (nao deixe o token exposto no notebook)
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="SEU_TOKEN_AQUI",
#     overwrite=True
# )


In [ ]:
RUN_ON_HARDWARE = False   # <- mude para True quando tiver as credenciais

if RUN_ON_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    from sklearn.metrics import accuracy_score

    # Conecta ao IBM Quantum e seleciona o backend com menor fila
    service = QiskitRuntimeService()
    backend = service.least_busy(operational=True, simulator=False)
    print("Backend selecionado:", backend.name)

    # Passo 2: Transpilar
    # optimization_level=3: maxima otimizacao (circuito mais raso = menos ruido)
    # O transpilador: (a) substitui portas pelas nativas do chip
    #                 (b) remapeia qubits para a topologia fisica
    #                 (c) minimiza numero de portas
    pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
    circuit_hw    = pm.run(circuit)
    observable_hw = observable.apply_layout(circuit_hw.layout)

    print("Profundidade do circuito transpilado:", circuit_hw.depth())
    n_cx = dict(circuit_hw.count_ops()).get("cx", 0) + dict(circuit_hw.count_ops()).get("ecr", 0)
    print("Portas de 2 qubits (principal fonte de ruido):", n_cx)

    def forward_hw(data, weights, est):
        # Mesmo que forward(), mas usa o circuito transpilado para o hardware
        data = np.atleast_2d(data)
        w = np.broadcast_to(weights, (data.shape[0], len(weights)))
        params = np.concatenate([data, w], axis=1)
        res = est.run([(circuit_hw, observable_hw, params)]).result()[0]
        return np.asarray(res.data.evs)

    # Passo 3: Executar com shots reais
    # shots=4096: numero de execucoes do circuito para estimar o valor esperado
    # Mais shots = mais precisao, mas mais tempo de QPU
    estimator_hw = EstimatorV2(mode=backend)
    estimator_hw.options.default_shots = 4096

    preds_hw = np.where(forward_hw(X_test, trained_weights, estimator_hw) >= 0, 1, -1)
    acc_hw   = accuracy_score(y_test, preds_hw)
    acc_sim  = accuracy_score(y_test, preds_test)

    print(f"\nAcuracia no simulador (referencia): {acc_sim*100:.1f}%")
    print(f"Acuracia no hardware (real):        {acc_hw*100:.1f}%")
    print(f"Queda por ruido:                    {(acc_sim - acc_hw)*100:.1f} p.p.")
    # E esperado queda de 5-15% por decoerencia e erros de porta
    # Para reduzir: ansatz mais raso, dynamical decoupling, mitigacao de erros

else:
    print("RUN_ON_HARDWARE = False — hardware nao executado.")
    print("Configure seu token IBM e defina RUN_ON_HARDWARE = True.")


## 8.1 Salvando os resultados do hardware automaticamente

Esta celula salva tudo que foi gerado — simulador e hardware — em um arquivo `.txt` formatado.
Serve como evidencia da execucao para o relatorio e para a apresentacao.


In [ ]:
import json
import datetime
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ── Coleta os resultados do simulador (sempre disponiveis) ──
acc_train = accuracy_score(y_train, preds_train)
acc_test  = accuracy_score(y_test,  preds_test)
report    = classification_report(y_test, preds_test,
               target_names=["nao-diabetica (-1)", "diabetica (+1)"])
cm        = confusion_matrix(y_test, preds_test, labels=[-1, 1])

# ── Monta o conteudo do relatorio ──
lines = []
lines.append("=" * 60)
lines.append("  RESULTADO — VQC DIABETES QUANTUM CLASSIFIER")
lines.append("=" * 60)
lines.append(f"  Data/hora: {datetime.datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
lines.append(f"  Dataset:   Pima Indians Diabetes (768 amostras, 2 features)")
lines.append(f"  Features:  Glucose x BMI (normalizados para [0, pi])")
lines.append(f"  Circuito:  z_feature_map (reps=1) + efficient_su2 (reps=2)")
lines.append(f"  Qubits:    2")
lines.append(f"  Parametros treinaveis: {n_weight}")
lines.append(f"  Amostras de treino usadas: {len(X_train_small)}")
lines.append(f"  Iteracoes do otimizador (COBYLA): {len(history)}")
lines.append(f"  Custo final (MSE): {history[-1]:.4f}")
lines.append("")
lines.append("-" * 60)
lines.append("  SIMULADOR (StatevectorEstimator — sem ruido)")
lines.append("-" * 60)
lines.append(f"  Acuracia treino : {acc_train*100:.1f}%")
lines.append(f"  Acuracia teste  : {acc_test*100:.1f}%")
lines.append("")
lines.append("  Relatorio completo (teste):")
for l in report.strip().split("\n"):
    lines.append("    " + l)
lines.append("")
lines.append("  Matriz de confusao (teste):")
lines.append("                   Previsto")
lines.append("                nao-diab  diabet")
lines.append(f"  Real nao-diab   {cm[0,0]:5d}   {cm[0,1]:5d}")
lines.append(f"  Real diabet     {cm[1,0]:5d}   {cm[1,1]:5d}")
lines.append("")

# ── Adiciona resultado do hardware se disponivel ──
if RUN_ON_HARDWARE and 'preds_hw' in dir():
    acc_hw_val  = accuracy_score(y_test, preds_hw)
    report_hw   = classification_report(y_test, preds_hw,
                      target_names=["nao-diabetica (-1)", "diabetica (+1)"])
    cm_hw       = confusion_matrix(y_test, preds_hw, labels=[-1, 1])

    lines.append("-" * 60)
    lines.append(f"  HARDWARE IBM QUANTUM (backend: {backend.name})")
    lines.append(f"  shots = 4096")
    lines.append("-" * 60)
    lines.append(f"  Acuracia teste  : {acc_hw_val*100:.1f}%")
    lines.append(f"  Queda vs simulador: {(acc_test - acc_hw_val)*100:.1f} p.p.")
    lines.append("")
    lines.append("  Relatorio completo (teste — hardware):")
    for l in report_hw.strip().split("\n"):
        lines.append("    " + l)
    lines.append("")
    lines.append("  Matriz de confusao (teste — hardware):")
    lines.append("                   Previsto")
    lines.append("                nao-diab  diabet")
    lines.append(f"  Real nao-diab   {cm_hw[0,0]:5d}   {cm_hw[0,1]:5d}")
    lines.append(f"  Real diabet     {cm_hw[1,0]:5d}   {cm_hw[1,1]:5d}")
    lines.append("")
else:
    lines.append("-" * 60)
    lines.append("  HARDWARE IBM QUANTUM")
    lines.append("-" * 60)
    lines.append("  Nao executado (RUN_ON_HARDWARE = False)")
    lines.append("  Para rodar no hardware: defina RUN_ON_HARDWARE = True")
    lines.append("  e configure seu token IBM na Secao 8.")
    lines.append("")

lines.append("-" * 60)
lines.append("  PESOS TREINADOS (trained_weights)")
lines.append("-" * 60)
for idx, w in enumerate(trained_weights):
    lines.append(f"  theta[{idx:02d}] = {w:.6f}")
lines.append("")
lines.append("=" * 60)
lines.append("  Arquivo gerado automaticamente pelo notebook VQC Diabetes")
lines.append("=" * 60)

texto = "\n".join(lines)

# ── Salva em arquivo .txt ──
output_path = Path("resultado_vqc_diabetes.txt")
output_path.write_text(texto, encoding="utf-8")

print(texto)
print()
print(f"Arquivo salvo em: {output_path.resolve()}")


## 8.2 Salvando os graficos em alta resolucao

Regera e salva todos os graficos do notebook como `.png` para usar no relatorio e na apresentacao.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

graficos_dir = Path("graficos_vqc_diabetes")
graficos_dir.mkdir(exist_ok=True)

# ── 1. Curva de convergencia ──
fig1, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(range(len(history)), history, color="steelblue", linewidth=2)
ax1.set_xlabel("Iteracao", fontsize=12)
ax1.set_ylabel("Custo (MSE)", fontsize=12)
ax1.set_title("Convergencia do treinamento — VQC Diabetes", fontsize=13)
fig1.tight_layout()
fig1.savefig(graficos_dir / "01_convergencia.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: 01_convergencia.png")

# ── 2. Matriz de confusao ──
cm = confusion_matrix(y_test, preds_test, labels=[-1, 1])
fig2, ax2 = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["nao-diabetica", "diabetica"],
            yticklabels=["nao-diabetica", "diabetica"], ax=ax2,
            annot_kws={"size": 14})
ax2.set_xlabel("Previsto", fontsize=12)
ax2.set_ylabel("Real", fontsize=12)
ax2.set_title("Matriz de Confusao — VQC Diabetes (teste)", fontsize=12)
fig2.tight_layout()
fig2.savefig(graficos_dir / "02_matriz_confusao.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: 02_matriz_confusao.png")

# ── 3. Fronteira de decisao ──
gx, gy = np.meshgrid(np.linspace(0, float(np.pi), 60),
                     np.linspace(0, float(np.pi), 60))
grid = np.c_[gx.ravel(), gy.ravel()]
zz = forward(grid, trained_weights).reshape(gx.shape)

fig3, ax3 = plt.subplots(figsize=(8, 6))
cf = ax3.contourf(gx, gy, zz, levels=20, cmap="coolwarm", alpha=0.65)
plt.colorbar(cf, ax=ax3, label="<ZZ>  (vermelho=diabetica | azul=nao-diabetica)")
ax3.contour(gx, gy, zz, levels=[0.0], colors="k", linewidths=2)

for label, name, marker, color in [
    (-1, "nao-diabetica", "o", "steelblue"),
    ( 1, "diabetica",     "s", "tomato")
]:
    tr = y_train == label
    te = y_test  == label
    ax3.scatter(X_train[tr,0], X_train[tr,1], marker=marker, color=color,
                edgecolor="k", s=55, label=name + " (treino)")
    ax3.scatter(X_test[te,0],  X_test[te,1],  marker=marker, color="none",
                edgecolor=color, s=85, linewidths=1.8, label=name + " (teste)")

ax3.set_xlabel("Glucose (normalizado)", fontsize=12)
ax3.set_ylabel("BMI (normalizado)", fontsize=12)
ax3.set_title("Fronteira de Decisao — VQC Diabetes", fontsize=13)
ax3.legend(loc="upper left", fontsize=9)
fig3.tight_layout()
fig3.savefig(graficos_dir / "03_fronteira_decisao.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: 03_fronteira_decisao.png")

# ── 4. Comparacao simulador vs hardware (se disponivel) ──
if RUN_ON_HARDWARE and 'acc_hw_val' in dir():
    fig4, ax4 = plt.subplots(figsize=(6, 4))
    categorias = ["Simulador\n(sem ruido)", "Hardware IBM\n(com ruido)"]
    valores    = [acc_test * 100, acc_hw_val * 100]
    cores      = ["steelblue", "tomato"]
    bars = ax4.bar(categorias, valores, color=cores, edgecolor="k", width=0.4)
    for bar, val in zip(bars, valores):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{val:.1f}%", ha="center", va="bottom", fontsize=13, fontweight="bold")
    ax4.set_ylim(0, 105)
    ax4.set_ylabel("Acuracia (%)", fontsize=12)
    ax4.set_title("Simulador vs Hardware IBM — VQC Diabetes", fontsize=12)
    ax4.axhline(65, color="gray", linestyle="--", linewidth=1,
                label="Baseline (prever sempre nao-diabetica)")
    ax4.legend(fontsize=9)
    fig4.tight_layout()
    fig4.savefig(graficos_dir / "04_simulador_vs_hardware.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Salvo: 04_simulador_vs_hardware.png")
else:
    print("Grafico 04_simulador_vs_hardware.png sera gerado apos rodar no hardware.")

print()
print(f"Todos os graficos salvos na pasta: {graficos_dir.resolve()}")
print("Use esses arquivos diretamente no relatorio e na apresentacao.")


## 9. Recapitulacao e Discussao

### Qiskit Patterns aplicados

| Passo | O que fizemos |
|---|---|
| 1. Mapear | Codificamos Glucose e BMI com z_feature_map; ansatz efficient_su2; observavel ZZ |
| 2. Otimizar | Transpilamos circuito + observavel para o backend IBM (secao 8) |
| 3. Executar | StatevectorEstimator (treino) e EstimatorV2 (hardware) |
| 4. Pos-processar | Acuracia, sensibilidade, matriz de confusao, fronteira de decisao |

### Pontos-chave de QML

**Embedding importa:** comparamos z_feature_map vs zz_feature_map pelo kernel quantico.
"Mais entrelaçamento nao significa melhor" — para dados moderadamente separaveis, o circuito mais simples e suficiente e mais facil de treinar.

**Ordem dos parametros e critica:** ao combinar feature map + ansatz, os parametros precisam estar na ordem certa (dados antes dos pesos). Um erro aqui nao gera mensagem de erro — silenciosamente produz resultados errados.

**Simulador x hardware:** simulador e deterministico e sem ruido. Hardware tem ruido e fila.
Sempre desenvolva no simulador; leve ao hardware so o circuito final e raso.

**Barren plateaus:** circuitos muito profundos tem gradientes que desaparecem.
Com reps=2 e 12 parametros, ainda estamos em terreno seguro.

### Proximos passos sugeridos

- Usar os 8 atributos originais (8 qubits) e comparar com os 2 usados aqui
- Comparar com classicadores classicos (Regressao Logistica, SVM) no mesmo dataset
- Aplicar mitigacao de erros nas opcoes do EstimatorV2
- Explorar data reuploading: repetir o feature map entre camadas do ansatz

### Referencias

- IBM Quantum Learning — Use a quantum computer today e Quantum machine learning
- Havlicek et al., Supervised learning with quantum-enhanced feature spaces (Nature, 2019)
- Smith, J.W. et al., Using the ADAP learning algorithm to forecast the onset of diabetes mellitus (1988)
